# Unidade 1 · Do dado bruto ao pipeline
**Aprendizagem Computacional · 4º TIA · Apostila do Aluno · versão INICIAL**

Este notebook cobre os cap. 1 a 4, 6, 7, 17 e 23 da apostila. Vamos percorrer, num único conjunto de dados, o caminho: **pergunta → X e y → EDA → qualidade → separar → preparar dentro de um Pipeline → distâncias → PCA → anomalias**.

**Dados:** pinguins do arquipélago Palmer, na Antártida (344 aves, 3 espécies), carregados do `seaborn`.
**Pergunta:** dá para identificar a espécie de um pinguim a partir de medidas simples do corpo?

Regras: rode de cima para baixo; `random_state=42` em tudo que sorteia.

**Como usar a versão INICIAL:** procure os comentários `# TODO`. Complete o código e rode a célula. As células seguintes dependem das anteriores. Se travar, compare com a versão COMPLETO.

## 1 · Carregar e olhar a estrutura (cap. 2 e 4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

SEMENTE = 42
from pathlib import Path
CSV_LOCAL = Path('../dados/penguins.csv')   # cópia no repositório: funciona sem internet
if CSV_LOCAL.exists():
    pinguins = pd.read_csv(CSV_LOCAL)
else:
    pinguins = sns.load_dataset('penguins')   # Colab: baixa da internet
print(pinguins.shape)
pinguins.head()

In [ ]:
pinguins.info()

**Pergunta para discussão:** qual é a unidade de análise, qual é o y e que tipo de tarefa temos?

**Resposta esperada:** cada linha é um pinguim (a unidade de análise). O y é `species`, uma categoria com 3 valores, então a tarefa é **classificação multiclasse supervisionada**. As demais colunas são candidatas a X.

In [ ]:
ALVO = None          # TODO: nome da coluna alvo
NUMERICAS = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
CATEGORICAS = []     # TODO: quais colunas de texto entram em X?
X = pinguins[NUMERICAS + CATEGORICAS]
y = pinguins[ALVO]
print('X:', X.shape, '| y:', y.shape)

## 2 · Qualidade e distribuição (cap. 4)

In [ ]:
print('Ausentes por coluna:')
print(pinguins.isna().sum())
print()
print('Proporção de cada espécie:')
print(y.value_counts(normalize=True).round(3))

**Leitura:** há 2 linhas sem nenhuma medida e 11 sem o sexo. As classes não são iguais (Adelie é a maior, com cerca de 44%), mas nenhuma é rara. Isso importa para a régua mínima: um modelo que chuta sempre "Adelie" acerta cerca de 44%.

In [ ]:
pinguins.describe().T

In [ ]:
sns.pairplot(pinguins, hue=ALVO, vars=NUMERICAS, corner=True, plot_kws={'s': 12})
plt.show()

**Pergunta para discussão:** olhando o pairplot, qual par de variáveis parece separar bem as três espécies?

**Resposta esperada:** comprimento do bico × profundidade do bico (ou × comprimento da nadadeira) forma três nuvens quase separadas. É um bom sinal de que um classificador simples vai funcionar. Mas isso ainda é hipótese: só o teste confirma.

In [ ]:
corr = pinguins[NUMERICAS].corr().round(2)
sns.heatmap(corr, annot=True, cmap='RdBu_r', vmin=-1, vmax=1)
plt.title('Correlação entre medidas'); plt.show()

**Leitura:** nadadeira e massa têm correlação de 0,87. Elas carregam informação parecida (redundância), o que não é um erro, mas ajuda a explicar por que retirar uma delas quase não muda o resultado.

## 3 · Separar ANTES de preparar (cap. 5 e 6)
Retiramos as 2 linhas sem nenhuma medida (não há o que imputar) e separamos 25% para teste. O `stratify=y` mantém a proporção das espécies nos dois lados.

In [ ]:
from sklearn.model_selection import train_test_split

sem_medidas = pinguins[NUMERICAS].isna().all(axis=1)
X, y = X[~sem_medidas], y[~sem_medidas]
# TODO: separe 25% para teste, estratificando por y e com random_state=SEMENTE
X_treino, X_teste, y_treino, y_teste = ...
print('Treino:', X_treino.shape, '| Teste:', X_teste.shape)

## 4 · A régua mínima: baseline

In [ ]:
from sklearn.dummy import DummyClassifier
baseline = None  # TODO: DummyClassifier que sempre chuta a classe mais frequente, ajustado no treino
print(f'Acurácia do baseline no teste: {baseline.score(X_teste, y_teste):.3f}')

## 5 · Preparação dentro de um Pipeline (cap. 6)
Imputação, padronização e *one-hot* ficam dentro do `Pipeline`. Assim tudo é aprendido **só com o treino**.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.neighbors import KNeighborsClassifier

preparo = ColumnTransformer([
    ('num', Pipeline([('imputa', SimpleImputer(strategy='median')),
                      # TODO: acrescente a etapa de padronização aqui
                      ]), NUMERICAS),
    ('cat', Pipeline([('imputa', SimpleImputer(strategy='most_frequent')),
                      ('codifica', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAS),
])
knn_com_escala = Pipeline([('preparo', preparo), ('modelo', KNeighborsClassifier(n_neighbors=5))])
knn_com_escala.fit(X_treino, y_treino)
print(f'k-NN com Pipeline completo: {knn_com_escala.score(X_teste, y_teste):.3f}')

## 6 · Por que a escala importa (cap. 7)
Mesmo modelo, mesmos dados, **uma** diferença: sem padronizar. Mudamos uma coisa por vez para saber o que causou a diferença.

In [ ]:
preparo_sem_escala = ColumnTransformer([
    ('num', SimpleImputer(strategy='median'), NUMERICAS),
    ('cat', Pipeline([('imputa', SimpleImputer(strategy='most_frequent')),
                      ('codifica', OneHotEncoder(handle_unknown='ignore'))]), CATEGORICAS),
])
knn_sem_escala = Pipeline([('preparo', preparo_sem_escala), ('modelo', KNeighborsClassifier(n_neighbors=5))])
knn_sem_escala.fit(X_treino, y_treino)

resultado = pd.DataFrame({
    'configuração': ['Baseline (classe mais frequente)', 'k-NN sem padronizar', 'k-NN com padronização'],
    'acurácia no teste': [baseline.score(X_teste, y_teste), knn_sem_escala.score(X_teste, y_teste),
                          knn_com_escala.score(X_teste, y_teste)]})
resultado.round(3)

In [ ]:
from sklearn.metrics import pairwise_distances
dois = X_treino[NUMERICAS].dropna().iloc[:2]
print('Dois pinguins:'); print(dois)
dif = (dois.iloc[0] - dois.iloc[1]) ** 2
print('\nContribuição de cada coluna para a distância² SEM padronizar (%):')
print((100 * dif / dif.sum()).round(1))

**Leitura:** a padronização levou o k-NN de cerca de 80% para 100% de acerto no teste. Acertar 100% aqui não indica vazamento: as três espécies são de fato bem separadas por essas medidas, como o pairplot sugeria. **Em dados reais de projeto, 100% é sinal de alerta** e pede investigação (cap. 6).

Sem padronizar, a massa corporal (em gramas) responde por quase toda a distância. O k-NN passa a escolher vizinhos quase só pelo peso, e as medidas do bico, que separavam bem as espécies, ficam praticamente ignoradas.

**Pergunta para discussão:** se ajustássemos o `StandardScaler` na base inteira antes do `train_test_split`, o que mudaria?

**Resposta esperada:** a média e o desvio seriam calculados também com os pinguins do teste. Seria um vazamento pequeno nestes dados, mas é um vício de método que, em outros problemas, infla bastante o resultado. Com o `Pipeline`, o problema não existe.

## 7 · Olhar os erros
A acurácia é um número só. A matriz de confusão mostra **quem** foi confundido com quem.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_estimator(knn_com_escala, X_teste, y_teste, cmap='Blues')
plt.title('k-NN com padronização · teste'); plt.show()

## 8 · Redução de dimensionalidade: PCA (cap. 17)
Quatro medidas numéricas são quatro dimensões; não dá para desenhar. O PCA resume as quatro em componentes ordenados pela variância que guardam. Ajustamos **só no treino**, dentro de um pipeline.

In [ ]:
from sklearn.decomposition import PCA
from sklearn.pipeline import make_pipeline

X_treino_num = X_treino[NUMERICAS].dropna()
# TODO: pipeline com StandardScaler e PCA de 4 componentes, ajustado em X_treino_num
pca = ...
var = pca[-1].explained_variance_ratio_
print('Variância explicada por componente:', var.round(3))
print('Acumulada:', var.cumsum().round(3))
pd.DataFrame(pca[-1].components_[:2], index=['PC1', 'PC2'], columns=NUMERICAS).round(2)

In [ ]:
Z = pca.transform(X_treino_num)
especie = y_treino.loc[X_treino_num.index]
for nome in especie.unique():
    m = (especie == nome).to_numpy()
    plt.scatter(Z[m, 0], Z[m, 1], s=14, alpha=0.7, label=nome)
plt.xlabel(f'PC1 ({var[0]:.0%} da variância)'); plt.ylabel(f'PC2 ({var[1]:.0%})')
plt.legend(); plt.title('Pinguins do treino projetados em 2 componentes'); plt.show()

**Leitura:** os dois primeiros componentes guardam a maior parte da variância. O PC1 tem pesos altos em nadadeira, massa e comprimento do bico: é um eixo de "tamanho". No plano PC1 × PC2, as três espécies aparecem quase separadas, mesmo que **o PCA nunca tenha visto a espécie**: as cores foram pintadas depois.

**Pergunta para discussão:** se o objetivo fosse prever a espécie, deveríamos sempre trocar as 4 medidas pelos 2 componentes?

**Resposta esperada:** não necessariamente. O PCA maximiza variância, não a separação das classes. Compare as duas versões com o mesmo protocolo (cap. 5 e 10) antes de decidir. Com só 4 colunas, a redução raramente compensa a perda de interpretação.

## 9 · Detecção de anomalias (cap. 23)
Quais pinguins são os mais "estranhos" para as medidas da sua espécie? Usamos dois métodos: a regra do IQR, uma variável por vez, e o Isolation Forest, que considera todas juntas.

In [ ]:
def fora_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    return (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)

medidas = pinguins.dropna(subset=NUMERICAS).copy()
por_coluna = medidas.groupby(ALVO)[NUMERICAS].transform(fora_iqr)
print('Casos fora do IQR (dentro da própria espécie), por coluna:')
print(por_coluna.sum())

In [ ]:
from sklearn.ensemble import IsolationForest

escores = []
for nome, grupo in medidas.groupby(ALVO):
    Xg = StandardScaler().fit_transform(grupo[NUMERICAS])
    # TODO: IsolationForest com contamination=0.02 e random_state=SEMENTE, ajustado em Xg
    iso = ...
    escores.append(pd.Series(-iso.score_samples(Xg), index=grupo.index))
medidas['escore_anomalia'] = pd.concat(escores)
medidas.nlargest(6, 'escore_anomalia')[[ALVO] + NUMERICAS + ['escore_anomalia']].round(2)

**Pergunta para discussão:** devemos apagar esses pinguins antes de treinar o modelo?

**Resposta esperada:** não automaticamente. Nenhuma medida é fisicamente impossível; são aves reais, apenas grandes ou pequenas para a espécie. Apagar casos reais deixa o modelo despreparado para eles. Remover só se faz com erro comprovado, e a regra deve ser decidida com o treino. Num projeto de fiscalização, esta tabela seria justamente o produto: a lista do que verificar primeiro.

## 10 · Checklist da Unidade 1
- [ ] Sei dizer a unidade de análise, o y e a tarefa.
- [ ] Fiz EDA com perguntas: estrutura, qualidade, distribuição, relações.
- [ ] Separei treino e teste **antes** de preparar.
- [ ] Tenho um baseline.
- [ ] Toda preparação que usa estatística está dentro do `Pipeline`.
- [ ] Mudei uma coisa por vez ao comparar.
- [ ] Sei usar o PCA para visualizar e sei que ele não olha o y.
- [ ] Investigo anomalias antes de decidir apagá-las.

**No seu projeto (entrega de 10/11):** repita as seções 1 e 2 deste notebook com a base de vocês: estrutura, ausentes, distribuição do y e 3 gráficos de relação com y.